# bRAG: Basic (naive) RAG Implementation

This notebook demonstrates a complete implementation of a basic RAG system that enables question-answering over PDF documents. 
The implementation is model, database, and document loader agnostic, though it's currently configured with:
- LLM: DeepSeek (OpenAI-compatible API)
- Vector Database: Chroma (本地持久化存储，无需 Docker)
- Document Loader: PyPDFLoader

The system combines several key components:
1. Document Loading: Loads PDF documents (extensible to other document types)
2. Text Processing: Splits documents into manageable chunks
3. Vector Operations:
   - Embeds text using Ollama nomic-embed-text model
   - Stores vectors in local Chroma vector database (持久化到本地目录)
4. Retrieval System: Implements efficient document retrieval
5. LLM Integration: Uses DeepSeek model for generating responses

All components can be swapped out for alternatives (e.g., different LLMs, vector stores, or document loaders) 
while maintaining the same overall architecture.

This implementation serves as a foundation for building more complex RAG applications
and can be customized based on specific use cases.

----------------------------------------

## Pre-requisites (optional but recommended)

### Only do the first step if you have never created a virtual environment for this repository. Otherwise, make sure that the Python Kernel that you selected is from your `venv/` folder.

In [1]:
# Create virtual environment
! python -m venv venv

Error: [Errno 13] Permission denied: 'e:\\own study code\\agent-llm\\bRAG-langchain-wdl\\venv\\Scripts\\python.exe'


In [2]:
# Activate virtual Python environment
! source venv/bin/activate

'source' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [3]:
# If your Python is not from your venv path, ensure that your IDE's kernel selection (on the top right corner) is set to the correct path 
# (your path output should contain "...venv/bin/python")

! which python

'which' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [ ]:
# Install all packages
! pip install -r requirements.txt --quiet

## Environment

`(1) Packages`

In [5]:
import os
# python-dotenv 是一个常用的工具，它允许你从 .env 文件中读取环境变量，这对于管理配置信息（如 API 密钥、数据库连接字符串等）非常有用，特别是当你不想将这些敏感信息直接硬编码在代码中时。
# from dotenv import load_dotenv

# Load all environment variables from .env file
# load_dotenv()

# Access the environment variables
langchain_tracing_v2 = os.getenv('LANGCHAIN_TRACING_V2')
langchain_endpoint = os.getenv('LANGCHAIN_ENDPOINT')
langchain_api_key = os.getenv('LANGCHAIN_API_KEY')

## LLM - DeepSeek (OpenAI-compatible)
api_key = os.getenv('DEEPSEEK_API_KEY').strip()
base_url = os.getenv('DEEPSEEK_API_BASE').strip()

print('api_key----', api_key)
print('base_url----', base_url)

## Chroma 本地向量数据库配置
# Chroma 无需 Docker，数据直接持久化到本地目录
chroma_persist_dir = os.getenv('CHROMA_PERSIST_DIR', './chroma_db')  # 本地持久化目录
chroma_collection = os.getenv('CHROMA_COLLECTION', 'langchain_rag')  # collection 名称

print('chroma_persist_dir----', chroma_persist_dir)
print('chroma_collection----', chroma_collection)

api_key---- sk-16dbae3437bb41aca6a6804d503127ad
base_url---- https://api.deepseek.com
chroma_persist_dir---- ./chroma_db
chroma_collection---- langchain_rag


`(2) LangSmith`

https://docs.smith.langchain.com/

In [6]:
os.environ['LANGCHAIN_TRACING_V2'] = langchain_tracing_v2
os.environ['LANGCHAIN_ENDPOINT'] = langchain_endpoint
os.environ['LANGCHAIN_API_KEY'] = langchain_api_key

`(3) API Keys`

In [ ]:
# DeepSeek 使用 OpenAI 兼容接口，通过 base_url 指向 DeepSeek 服务
os.environ['OPENAI_API_KEY'] = api_key
os.environ['OPENAI_API_BASE'] = base_url
os.environ['MODEL_NAME'] = MODEL_NAME

`(4) Chroma 本地存储检查`

Chroma 无需启动任何外部服务，数据直接持久化到本地文件夹，开箱即用。

In [8]:
# 检查 Chroma 持久化目录
import os

def check_chroma_dir(persist_dir='./chroma_db'):
    """检查 Chroma 持久化目录状态"""
    if os.path.exists(persist_dir):
        print(f"✅ Chroma 持久化目录已存在：{os.path.abspath(persist_dir)}")
        files = os.listdir(persist_dir)
        print(f"   目录内容：{files if files else '(空目录，首次运行将自动创建数据文件)'}")
    else:
        print(f"📁 Chroma 持久化目录不存在，首次运行时将自动创建：{os.path.abspath(persist_dir)}")
    print("✅ Chroma 无需 Docker，无需任何外部服务，可直接使用！")

check_chroma_dir(chroma_persist_dir)

✅ Chroma 持久化目录已存在：e:\own study code\agent-llm\bRAG-langchain-wdl\chroma_db
   目录内容：['61a7d6f8-7b9c-459b-a65f-5e68b4ea181c', 'chroma.sqlite3']
✅ Chroma 无需 Docker，无需任何外部服务，可直接使用！


## Full RAG App (Basic)

In [ ]:
# 这段代码实现了一个 RAG（检索增强生成） 流程，用于从 PDF 文档中提取信息并基于 LLM（此处使用 DeepSeek 模型）回答问题。
from langchain_community.document_loaders import PyPDFLoader           # 用于加载PDF文档
from langchain_text_splitters import RecursiveCharacterTextSplitter    # 用于递归分割文本
from langchain_chroma import Chroma                                    # 用于 Chroma 向量存储
from langchain_core.output_parsers import StrOutputParser              # 用于解析输出为字符串
from langchain_core.runnables import RunnablePassthrough               # 用于传递运行时数据
from langchain_openai import ChatOpenAI                                # 用于 OpenAI 兼容聊天模型
from langchain_ollama import OllamaEmbeddings                          # 用于 Ollama 本地 Embedding 
from langchain_core.prompts import ChatPromptTemplate                  # 用于聊天提示模板

#### 1. 索引部分，用于文档处理和索引创建
pdf_file_path = "test/langchain_turing.pdf"   # 定义PDF文件路径
# pdf_file_path = "test/2026年周会任务.pdf"         # 定义PDF文件路径
loader = PyPDFLoader(pdf_file_path)              # 创建PDF加载器实例
# 语法：创建 PyPDFLoader 实例，调用 load() 方法返回文档列表（每个元素是一个 Document 对象，包含 page_content 和 metadata）
docs = loader.load()                             # 加载PDF文档内容

#### 2. Split 分割文本
# 作用：将长文档按 2000 字符分块，块间重叠 200 字符，保持语义连贯。
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
# 语法：split_documents() 接收文档列表，返回分割后的文档块列表。
splits = text_splitter.split_documents(docs)     # 使用文本分割器将文档分割成更小的块

print(f"文档块数量--0--: {len(splits)}")

#### 3. Embedding 模型
# 使用 Ollama 部署本地 Embedding 模型，无需 API Key，免费且离线可用

# 方案1：使用HuggingFaceEmbeddings
# embedding_model = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2",
# )
# 放弃原因：sentence-transformers依赖 PyTorch，因网络问题，访问download.pytorch.org 超时；

# 方案2：使用FastEmbedEmbeddings
# embedding_model = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
# 放弃原因：onnxruntime 的 DLL 加载失败，venv 环境问题。

# 方案3：使用OpenAIEmbeddings（DeepSeek 不支持 Embedding，已放弃）
# embedding_model = OpenAIEmbeddings(...)

# 方案4：使用Ollama部署本地模型（推荐，免费且无需 API Key）
# 前提：本地已安装 Ollama 并拉取模型：ollama pull nomic-embed-text
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"  # 向量维度：768，大小：274M
)

print(f"Embedding model--1--: {embedding_model}")
# emb = embedding_model.embed_query("test")

#### 4. 创建 Chroma 向量存储（本地持久化，无需 Docker）
# Chroma 优势：
# - 纯 Python 实现，pip install chromadb 即可，无需任何外部服务
# - 支持本地持久化，数据保存到指定目录，重启后数据不丢失
# - Windows 完全兼容，无 milvus-lite 的平台限制问题
# - 适合本地开发、测试和中小规模生产环境

# persist_directory：指定本地持久化目录，Chroma 会自动创建并管理该目录
# collection_name：逻辑分组名称，类似 Milvus 的 collection
vector_store = Chroma(
    embedding_function=embedding_model,       # 用于生成向量嵌入的模型
    persist_directory=chroma_persist_dir,     # 本地持久化目录（自动创建）
    collection_name=chroma_collection,        # collection 名称
)

print(f"vector_store--2--: {vector_store}")

# 将文档分割块写入 Chroma
# 生成自定义 ID（字符串格式，与文档块一一对应）
ids = [str(i + 1) for i in range(len(splits))]
print(f"自定义文档ID列表：{ids}")

# 批量插入文档（指定 ids 参数，便于后续按 ID 查询/更新/删除）
insert_result = vector_store.add_documents(
    documents=splits,
    ids=ids  # 绑定文档与ID的映射关系
)

#### 5. 创建检索器
retriever = vector_store.as_retriever()  # 创建检索器，用于后续从向量存储中检索相关文档


#### 6. 检索与生成部分（RETRIEVAL and GENERATION）

# 6.1 提示模板（Prompt Template）
# 定义一个模板字符串，用于构建提示，要求基于提供的上下文来回答问题
template = """Answer the question based only on the following context: 
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)  # 使用模板创建聊天提示模板

# 6.2 LLM 配置（使用 DeepSeek）
# LLM - 使用 DeepSeek，通过 ChatOpenAI 的 openai_api_base 指向 DeepSeek 接口
llm = ChatOpenAI(
    model_name=model_name,                 # 指定使用的模型名称
    temperature=0.1,                           # 设置温度参数，控制输出的随机性，值越小输出越确定
    openai_api_key=api_key,                    # 设置 API 密钥
    openai_api_base=base_url                   # 设置 API 基础 URL，指向 DeepSeek
)

print(f"LLM model--4--: {llm}")

# 6.3 后处理函数（Post-processing）
# 将检索结果拼接到提示模板的 {context} 位置
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 6.4 RAG 链（Chain）
# 使用 LangChain LCEL（LangChain Expression Language）语法，通过管道符 | 串联组件：
# - retriever 检索相关文档 → format_docs 格式化为字符串 → 填入 {context}
# - RunnablePassthrough() 直接传递用户输入的问题 → 填入 {question}
# - prompt 格式化完整提示 → llm 生成回答 → StrOutputParser 解析为字符串
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(f"RAG chain--5--: {rag_chain}")

# 总结
# 该代码演示了一个完整的 RAG 流程：
# 加载 PDF → 分割文档 → 生成向量并存入本地 Chroma。
# 构建提示模板，配置 LLM（通过 OpenAI 兼容接口调用 DeepSeek）。
# 使用 LCEL 将检索器、提示模板、LLM 和输出解析器串联成链。
# 最终 rag_chain 可用来回答基于文档内容的问题。

e:\own study code\agent-llm\bRAG-langchain-wdl\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


文档块数量--0--: 27
Embedding model--1--: base_url='http://localhost:11434' model='nomic-embed-text' embed_instruction='passage: ' query_instruction='query: ' mirostat=None mirostat_eta=None mirostat_tau=None num_ctx=None num_gpu=None num_thread=None repeat_last_n=None repeat_penalty=None temperature=None stop=None tfs_z=None top_k=None top_p=None show_progress=False headers=None model_kwargs=None
vector_store--2--: <langchain_chroma.vectorstores.Chroma object at 0x000001B536782B50>
自定义文档ID列表：['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27']


C:\Users\wdlhao\AppData\Local\Temp\ipykernel_24872\841937759.py:44: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding_model = OllamaEmbeddings(


LLM model--4--: output_version=None client=<openai.resources.chat.completions.completions.Completions object at 0x000001B536780B10> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B53678E250> root_client=<openai.OpenAI object at 0x000001B535092450> root_async_client=<openai.AsyncOpenAI object at 0x000001B5384E7E90> model_name='deepseek-chat' temperature=0.1 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.deepseek.com' openai_proxy=None
RAG chain--5--: first={
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B536782B50>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], i

In [10]:
# Question
from pprint import pprint

pprint(rag_chain.invoke("What is this document about?"))

('Based solely on the provided context, this document is about **LangChain and '
 'its components, specifically LangGraph**, which is a framework for building '
 "applications with large language models (LLMs). It describes LangGraph's "
 'workflow design, its platform for production deployment, and features like '
 'defining nodes/edges, streaming, and support for multi-modal data.')
